# **Gold Layer**

In [1]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
#install on new fabric env
import reverse_geocoder as rg


StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 5, Finished, Available, Finished, False)

In [2]:
# from datetime import date,timedelta

# start_date= date.today() - timedelta(7)

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 6, Finished, Available, Finished, False)

In [3]:
df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 7, Finished, Available, Finished, False)

In [ ]:
# coordinates=(64.481,-148.637)
# rg.search(coordinates)[0]

In [ ]:
# rg.search(coordinates)[0].get('cc')

In [11]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_code(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get('cc')

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 15, Finished, Available, Finished, False)

In [12]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 16, Finished, Available, Finished, False)

In [13]:
# adding country_code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 17, Finished, Available, Finished, False)

In [15]:
# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 19, Finished, Available, Finished, False)

In [17]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')

StatementMeta(, 62a69c60-01c4-4c87-b531-29bcedb572f9, 21, Finished, Available, Finished, False)